In [1]:
import torch, os
from transformers import AutoModelForCausalLM

base_ckpt   = "meta-llama/Llama-2-7b-chat-hf"   # 原始未剪枝 checkpoint
mask_path   = "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/out/llama2-7b-chat-hf/unstructured/wanda_weightonly/GSM8K_cot0shot_goldreason/FT_mask/mask_bottom_0.100.pt"
pruned_dir  = "./tmp/llama2-7b-chat-pruned"      # 输出目录

# 1. 载入模型 & 掩码
model = AutoModelForCausalLM.from_pretrained(
    base_ckpt,
    torch_dtype=torch.float16,
    device_map="auto",          # 如果只想在 CPU 上做，可删掉
)
# mask  = torch.load(mask_path, map_location="cpu")   # dict[name] -> bool tensor

# is_top_mask = os.path.basename(mask_path).startswith("mask_top_")

# # 2. 把掩码应用到每个有 weight 的 module
# for name, module in model.named_modules():
#     if hasattr(module, "weight") and name in mask:
#         m = mask[name].to(module.weight.device)     # bool 同形状
#         if is_top_mask:
#             # True = 被剪掉 → 置 0
#             module.weight.data[m] = 0
#         else:
#             # True = 保留   → 反向置 0
#             module.weight.data[~m] = 0

# # 3. 保存一个新的 checkpoint；以后 vLLM/HF 都能直接从这里加载
# model.save_pretrained(pruned_dir)
# print(f"Pruned model saved to {pruned_dir}")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
for name, module in model.named_modules():
    print(module)
    break

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): LlamaRMSNorm()
  )
  (lm_

In [4]:
import torch.nn as nn
def find_layers(module, layers=[nn.Linear], name=""):
    """
    Recursively find the layers of a certain type in a module.

    Args:
        module (nn.Module): PyTorch module.
        layers (list): List of layer types to find.
        name (str): Name of the module.

    Returns:
        dict: Dictionary of layers of the given type(s) within the module.
    """
    if type(module) in layers:
        return {name: module}
    res = {}
    for name1, child in module.named_children():
        res.update(
            find_layers(
                child, layers=layers, name=name + "." + name1 if name != "" else name1
            )
        )
    return res

def check_sparsity(model):
    use_cache = model.config.use_cache
    model.config.use_cache = False

    layers = model.model.layers
    count = 0
    total_params = 0
    for i in range(len(layers)):
        layer = layers[i]
        subset = find_layers(layer)

        sub_count = 0
        sub_params = 0
        for name in subset:
            W = subset[name].weight.data
            count += (W == 0).sum().item()
            total_params += W.numel()

            sub_count += (W == 0).sum().item()
            sub_params += W.numel()

        print(f"layer {i} sparsity {float(sub_count)/sub_params:.6f}")

    model.config.use_cache = use_cache
    return float(count) / total_params 

In [5]:
check_sparsity(model)

layer 0 sparsity 0.099870
layer 1 sparsity 0.099870
layer 2 sparsity 0.099870
layer 3 sparsity 0.099870
layer 4 sparsity 0.099870
layer 5 sparsity 0.099870
layer 6 sparsity 0.099870
layer 7 sparsity 0.099870
layer 8 sparsity 0.099870
layer 9 sparsity 0.099870
layer 10 sparsity 0.099870
layer 11 sparsity 0.099870
layer 12 sparsity 0.099870
layer 13 sparsity 0.099870
layer 14 sparsity 0.099870
layer 15 sparsity 0.099870
layer 16 sparsity 0.099870
layer 17 sparsity 0.099870
layer 18 sparsity 0.099870
layer 19 sparsity 0.099870
layer 20 sparsity 0.099870
layer 21 sparsity 0.099870
layer 22 sparsity 0.099870
layer 23 sparsity 0.099870
layer 24 sparsity 0.099870
layer 25 sparsity 0.099870
layer 26 sparsity 0.099870
layer 27 sparsity 0.099870
layer 28 sparsity 0.099870
layer 29 sparsity 0.099870
layer 30 sparsity 0.099870
layer 31 sparsity 0.099870


0.09986996033031088

In [ ]:
import torch, os
from transformers import AutoModelForCausalLM
import torch.nn.utils.prune as prune

base_ckpt   = "meta-llama/Llama-2-7b-chat-hf"   # 原始未剪枝 checkpoint
mask_path   = "/common/users/sl2148/Public/yang_ouyang/alignment-attribution-code/out/llama2-7b-chat-hf/unstructured/wanda_weightonly/GSM8K_cot0shot_goldreason/FT_mask/mask_bottom_0.100.pt"
pruned_dir  = "/tmp/llama2-7b-chat-pruned"      # 输出目录

# 1. 载入模型 & 掩码
model = AutoModelForCausalLM.from_pretrained(
    base_ckpt,
    torch_dtype=torch.float16,
    device_map="auto",          # 如果只想在 CPU 上做，可删掉
)
mask  = torch.load(mask_path, map_location="cpu")   # dict[name] -> bool tensor

is_top_mask = os.path.basename(mask_path).startswith("mask_top_")

# 2. 把掩码应用到每个有 weight 的 module

for name, module in model.named_modules():
    if hasattr(module, "weight") and name in mask:
        m = mask[name].to(module.weight.device)

        # prune.custom_from_mask 会把 pruned weight 存在 module.weight_mask,
        # 并在 forward 时自动做乘法
        prune.custom_from_mask(module, name='weight', mask=~m if is_top_mask else m)

        # 如果你不需要保持 re-param 结构，直接把 mask 烧进去再删除 hook：
        prune.remove(module, 'weight')   # 现在 module.weight 已经被置 0
        
# 3. 保存一个新的 checkpoint；以后 vLLM/HF 都能直接从这里加载
model.save_pretrained(pruned_dir)
print(f"Pruned model saved to {pruned_dir}")
